<!-- ARTHASAATHI-PROVENANCE -->
> ## ⚠️ This notebook is demo evidence, not the running system
>
> The logic below was migrated to **`ml/src/decision/query_funnel.py`** and that module is what the application actually executes.
>
> **Role in the system:** Decision Layer -> Query Funnel
>
> This notebook is preserved because its stored outputs are the record of the original analysis. **Editing the cells below will not change the system's behaviour** — change the module and its tests instead.
>
> Tests for this logic live under `tests/`, and every agent is covered by the cross-cutting contract suite in `tests/common/test_contracts.py`.


# 🔻 Query Understanding Funnel
One question in → narrows step by step → one grounded answer out.
Two LLM calls total, regardless of question complexity: **one to plan, one to explain.**

```
                         Question
                            │
                            ▼
              ┌──────────────────────────────┐
              │   LLM CALL 1 — Planner        │   sees ONLY the question,
              │   (no data shown yet)         │   never the dataset
              └──────────────────────────────┘
                            │
                 outputs ONE structured plan:
            { is_specific, domain, fields, record_filter, computation }
                            │
              ┌─────────────┴─────────────┐
              │                           │
        is_specific=False           is_specific=True
              │                           │
              ▼                           ▼
      skip data entirely        ┌───────────────────┐
      (no narrowing needed)     │  CODE — narrowing  │   pure Python,
              │                 │  funnel:           │   deterministic,
              │                 │  domain → fields   │   no LLM call
              │                 │  → records         │
              │                 │  → compute (code)  │
              │                 └───────────────────┘
              │                           │
              └─────────────┬─────────────┘
                            ▼
              ┌──────────────────────────────┐
              │   LLM CALL 2 — Explainer      │   general Q → answers with
              │                               │   NO data attached
              │                               │   specific Q → answers using
              │                               │   ONLY the narrowed slice
              └──────────────────────────────┘
                            │
                            ▼
                       Final Answer
```

### Why this shape
- **Planning and answering are different jobs — split into two calls, not one.** The planner never sees the data (cheap, fast, can't be distracted by row content). The explainer never re-decides scope (it just explains what it's given).
- **The narrowing itself (domain → fields → records) is pure code, not an LLM call.** Once the plan says "domain=credit_card, fields=[issuer, annual_fee], filter=annual_fee>1000", that's a deterministic filter — no reason to spend a model call on it.
- **General vs specific is a real fork, not a formality.** A general question ("what's a good rule of thumb for choosing a card?") skips the dataset completely — it gets answered like a normal LLM call, no rows attached, no wasted context. A specific question ("which credit cards have annual fee under 1000?") goes through the full funnel and the explainer only ever sees the narrowed slice.
- **One plan covers any question shape.** You don't write new logic per question type — the planner's structured output (`domain`, `fields`, `record_filter`, `computation`) is general enough to express both "show me X" and "what's the average Y for Z" without bespoke code paths.


In [1]:
import requests
import json, re, textwrap, requests
from pathlib import Path
from typing import Any, Optional

print("✅ Imports OK")


✅ Imports OK


## Step 1 — Input Data
One file, 150 rows, grouped into domains by content (~20 rows credit_card, ~20 asset_management,
~20 insurance, etc.). **Replace this cell with your real file load** — the funnel only needs each
row to carry a `domain` field plus whatever other fields are relevant.

In [2]:
import json

STRATEGIES = [
    {
        "strategy_id": "S1",
        "strategy_name": "Conservative Recovery",

        "emergency_fund": {
            "target_months": 12,
            "target_amount": 600000,
            "monthly_contribution": 15000,
            "completion_months": 40
        },

        "debt_plan": {
            "current_debt": 400000,
            "debt_type": "Personal Loan",
            "interest_rate": 14.5,
            "monthly_payment": 18000,
            "closure_months": 24
        },

        "insurance": {
            "term_cover": 10000000,
            "health_cover": 1000000,
            "annual_premium": 25000
        },

        "goals": [
            {
                "goal_name": "Emergency Security",
                "target_amount": 600000,
                "target_year": 2028
            }
        ],

        "investments": [
            {"instrument": "Liquid Fund", "allocation_pct": 40},
            {"instrument": "Corporate Bond Fund", "allocation_pct": 35},
            {"instrument": "Gold ETF", "allocation_pct": 15},
            {"instrument": "Nifty 50 Index Fund", "allocation_pct": 10}
        ],

        "expected_return_pct": 8.2,
        "volatility_pct": 5.0,
        "max_drawdown_pct": 8.0,
        "retirement_success_probability": 72,
        "goal_success_probability": 91,
        "expected_networth_20_years": 24000000
    },

    {
        "strategy_id": "S2",
        "strategy_name": "Balanced Wealth Builder",

        "emergency_fund": {
            "target_months": 6,
            "target_amount": 300000,
            "monthly_contribution": 8000,
            "completion_months": 18
        },

        "debt_plan": {
            "current_debt": 250000,
            "debt_type": "Personal Loan",
            "interest_rate": 12.5,
            "monthly_payment": 12000,
            "closure_months": 20
        },

        "insurance": {
            "term_cover": 15000000,
            "health_cover": 1500000,
            "annual_premium": 32000
        },

        "goals": [
            {
                "goal_name": "House Down Payment",
                "target_amount": 2500000,
                "target_year": 2030
            },
            {
                "goal_name": "Retirement",
                "target_amount": 40000000,
                "target_year": 2055
            }
        ],

        "investments": [
            {"instrument": "Nifty 50 Index Fund", "allocation_pct": 30},
            {"instrument": "Flexi Cap Fund", "allocation_pct": 25},
            {"instrument": "Corporate Bond Fund", "allocation_pct": 25},
            {"instrument": "Gold ETF", "allocation_pct": 10},
            {"instrument": "Liquid Fund", "allocation_pct": 10}
        ],

        "expected_return_pct": 11.5,
        "volatility_pct": 10.0,
        "max_drawdown_pct": 15.0,
        "retirement_success_probability": 84,
        "goal_success_probability": 86,
        "expected_networth_20_years": 42000000
    },

    {
        "strategy_id": "S3",
        "strategy_name": "Goal Accelerator",

        "emergency_fund": {
            "target_months": 6,
            "target_amount": 300000,
            "monthly_contribution": 5000,
            "completion_months": 24
        },

        "debt_plan": {
            "current_debt": 150000,
            "debt_type": "Auto Loan",
            "interest_rate": 9.5,
            "monthly_payment": 10000,
            "closure_months": 16
        },

        "insurance": {
            "term_cover": 15000000,
            "health_cover": 1500000,
            "annual_premium": 30000
        },

        "goals": [
            {
                "goal_name": "House Purchase",
                "target_amount": 5000000,
                "target_year": 2032
            },
            {
                "goal_name": "Child Education",
                "target_amount": 3000000,
                "target_year": 2038
            }
        ],

        "investments": [
            {"instrument": "Flexi Cap Fund", "allocation_pct": 35},
            {"instrument": "Nifty 50 Index Fund", "allocation_pct": 25},
            {"instrument": "Mid Cap Fund", "allocation_pct": 25},
            {"instrument": "Gold ETF", "allocation_pct": 15}
        ],

        "expected_return_pct": 13.8,
        "volatility_pct": 18.0,
        "max_drawdown_pct": 25.0,
        "retirement_success_probability": 88,
        "goal_success_probability": 93,
        "expected_networth_20_years": 56000000
    },

    {
        "strategy_id": "S4",
        "strategy_name": "Aggressive Wealth Maximizer",

        "emergency_fund": {
            "target_months": 3,
            "target_amount": 150000,
            "monthly_contribution": 3000,
            "completion_months": 12
        },

        "debt_plan": {
            "current_debt": 100000,
            "debt_type": "Education Loan",
            "interest_rate": 8.0,
            "monthly_payment": 8000,
            "closure_months": 12
        },

        "insurance": {
            "term_cover": 20000000,
            "health_cover": 2000000,
            "annual_premium": 42000
        },

        "goals": [
            {
                "goal_name": "Financial Independence",
                "target_amount": 100000000,
                "target_year": 2045
            }
        ],

        "investments": [
            {"instrument": "Small Cap Fund", "allocation_pct": 40},
            {"instrument": "Mid Cap Fund", "allocation_pct": 30},
            {"instrument": "NASDAQ ETF", "allocation_pct": 20},
            {"instrument": "Gold ETF", "allocation_pct": 10}
        ],

        "expected_return_pct": 18.0,
        "volatility_pct": 30.0,
        "max_drawdown_pct": 45.0,
        "retirement_success_probability": 83,
        "goal_success_probability": 87,
        "expected_networth_20_years": 95000000
    },

    {
        "strategy_id": "S5",
        "strategy_name": "Tax Optimized Retirement",

        "emergency_fund": {
            "target_months": 6,
            "target_amount": 300000,
            "monthly_contribution": 7000,
            "completion_months": 20
        },

        "debt_plan": {
            "current_debt": 200000,
            "debt_type": "Home Loan",
            "interest_rate": 8.5,
            "monthly_payment": 15000,
            "closure_months": 180
        },

        "insurance": {
            "term_cover": 15000000,
            "health_cover": 1500000,
            "annual_premium": 30000
        },

        "goals": [
            {
                "goal_name": "Retirement",
                "target_amount": 50000000,
                "target_year": 2055
            }
        ],

        "investments": [
            {"instrument": "ELSS Fund", "allocation_pct": 35},
            {"instrument": "NPS", "allocation_pct": 25},
            {"instrument": "PPF", "allocation_pct": 20},
            {"instrument": "Nifty 50 Index Fund", "allocation_pct": 10},
            {"instrument": "Gold ETF", "allocation_pct": 10}
        ],

        "expected_return_pct": 12.3,
        "volatility_pct": 11.0,
        "max_drawdown_pct": 18.0,
        "retirement_success_probability": 91,
        "goal_success_probability": 85,
        "expected_networth_20_years": 51000000
    }
]

print(json.dumps(STRATEGIES, indent=2))

STRATEGY_ROWS = [
    {
        "row_id": i,
        "domain": "strategy",
        **strategy
    }
    for i, strategy in enumerate(STRATEGIES)
]

[
  {
    "strategy_id": "S1",
    "strategy_name": "Conservative Recovery",
    "emergency_fund": {
      "target_months": 12,
      "target_amount": 600000,
      "monthly_contribution": 15000,
      "completion_months": 40
    },
    "debt_plan": {
      "current_debt": 400000,
      "debt_type": "Personal Loan",
      "interest_rate": 14.5,
      "monthly_payment": 18000,
      "closure_months": 24
    },
    "insurance": {
      "term_cover": 10000000,
      "health_cover": 1000000,
      "annual_premium": 25000
    },
    "goals": [
      {
        "goal_name": "Emergency Security",
        "target_amount": 600000,
        "target_year": 2028
      }
    ],
    "investments": [
      {
        "instrument": "Liquid Fund",
        "allocation_pct": 40
      },
      {
        "instrument": "Corporate Bond Fund",
        "allocation_pct": 35
      },
      {
        "instrument": "Gold ETF",
        "allocation_pct": 15
      },
      {
        "instrument": "Nifty 50 Index Fund",


## Stage 1 — Planner (LLM call #1, sees the question only — never the data)
Outputs a single structured plan. Decides general-vs-specific itself, as part of this call.
Schema discovery (`available_domains`, `available_fields`) is given so the planner can only
reference things that actually exist in the dataset — not from training-data guesses.

In [3]:
import json

def discover_schema(strategies):
    schema = {}

    for strategy in strategies:

        for key, value in strategy.items():

            if isinstance(value, dict):

                schema[key] = sorted(value.keys())

            elif (
                isinstance(value, list)
                and len(value) > 0
                and isinstance(value[0], dict)
            ):

                fields = set()

                for item in value:
                    fields.update(item.keys())

                schema[key] = sorted(fields)

            else:

                schema[key] = type(value).__name__

    return schema


SCHEMA = discover_schema(STRATEGIES)

print(json.dumps(SCHEMA, indent=2))

{
  "strategy_id": "str",
  "strategy_name": "str",
  "emergency_fund": [
    "completion_months",
    "monthly_contribution",
    "target_amount",
    "target_months"
  ],
  "debt_plan": [
    "closure_months",
    "current_debt",
    "debt_type",
    "interest_rate",
    "monthly_payment"
  ],
  "insurance": [
    "annual_premium",
    "health_cover",
    "term_cover"
  ],
  "goals": [
    "goal_name",
    "target_amount",
    "target_year"
  ],
  "investments": [
    "allocation_pct",
    "instrument"
  ],
  "expected_return_pct": "float",
  "volatility_pct": "float",
  "max_drawdown_pct": "float",
  "retirement_success_probability": "int",
  "goal_success_probability": "int",
  "expected_networth_20_years": "int"
}


In [18]:
from groq import Groq
import json
import os
from dotenv import load_dotenv
load_dotenv()
client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)
PLANNER_SYSTEM_PROMPT = """
You are a query-planning agent.

You will be given:

1. A USER QUESTION
2. The AVAILABLE SCHEMA of a dataset

You NEVER see actual records.
You NEVER answer the question.
Your only job is to produce a retrieval/computation plan.

AVAILABLE SCHEMA:
{schema}

--------------------------------------------------
CLASSIFICATION RULES
--------------------------------------------------

Set "is_specific" = true if answering the question requires
looking at dataset records.

Examples:

- Which strategy is best?
- Which option is safest?
- Which strategy is more aggressive?
- Which one should I choose?
- Which strategy has highest returns?
- Which plan has lowest risk?
- Compare the available strategies.
- Rank the strategies.
- Which option is good for retirement?
- Which option is balanced?
- I want a safe option.
- I don't want a lot of headache.
- I want stable returns.
- Which one is most suitable for me?

These ALL require dataset analysis and MUST return:

"is_specific": true

--------------------------------------------------

Set "is_specific" = false ONLY when the question can be
answered without any dataset records.

Examples:

- What is a mutual fund?
- What is volatility?
- Explain drawdown.
- What is asset allocation?
- What does CAGR mean?
- Explain retirement planning.

--------------------------------------------------
PLANNING RULES
--------------------------------------------------

If is_specific = true:

Return:

- domains
- fields
- record_filter
- computation
- computation_field

Rules:

1. Select ONLY fields needed to answer the question.

2. Use:

"record_filter": "all records"

when the user is comparing or choosing among options.

3. Use:

"computation": "compare"

for questions involving:

- best
- safest
- recommend
- choose
- rank
- compare
- aggressive
- conservative
- balanced
- suitable
- highest
- lowest

4. Use:

"max" for explicit highest-value requests.

Example:
"Which strategy has highest expected return?"

5. Use:

"min" for explicit lowest-value requests.

Example:
"Which strategy has lowest volatility?"

6. NEVER invent field names.

Only use fields that exist in the schema.

7. NEVER treat field names as domains.

Domains are logical dataset groups.
Fields are attributes inside domains.

--------------------------------------------------
OUTPUT FORMAT
--------------------------------------------------

Return STRICT JSON ONLY.

{{
  "is_specific": true,
  "reasoning": "...",
  "domains": [...],
  "fields": [...],
  "record_filter": "...",
  "computation": "...",
  "computation_field": "..."
}}

OR

{{
  "is_specific": false,
  "reasoning": "...",
  "domains": null,
  "fields": null,
  "record_filter": null,
  "computation": null,
  "computation_field": null
}}

No markdown.
No explanations.
No prose outside JSON.
"""
def plan_query(
    question: str,
    schema: dict,
    model: str = "llama-3.3-70b-versatile"
) -> dict:

    system_prompt = PLANNER_SYSTEM_PROMPT.format(
        schema=json.dumps(schema, indent=2)
    )

    try:

        completion = client.chat.completions.create(
            model=model,
            temperature=0,
            response_format={"type": "json_object"},
            messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": question
                }
            ]
        )

        text = completion.choices[0].message.content.strip()

        plan = json.loads(text)

    except Exception as e:

        print(f"⚠️ Planning failed: {e}")

        plan = {
            "is_specific": True,
            "reasoning": "fallback: planner call failed",
            "domains": list(schema.keys()),
            "fields": None,
            "record_filter": "all records",
            "computation": "none",
            "computation_field": None,
        }

    return plan

## Stage 2 — Narrowing Funnel (pure code, no LLM)
Executes the plan deterministically: **domain → fields → records → compute**.
The `record_filter` is plain-language from the planner, so this stage uses a small second
LLM-free heuristic parser for common comparison patterns, with a safe "include everything"
fallback when the filter can't be parsed mechanically — never silently drops data it can't
understand.

In [19]:
def filter_by_domain(rows: list[dict], domains: list[str]) -> list[dict]:
    """
    Safe domain filtering.

    If planner returns invalid domains (e.g. field names instead of domains),
    do NOT drop all rows.
    """

    if not domains:
        return rows

    available_domains = {r.get("domain") for r in rows}

    valid_domains = [
        d for d in domains
        if d in available_domains
    ]

    if not valid_domains:
        print(
            f"⚠️ Planner returned invalid domains {domains}. "
            f"Available domains: {available_domains}. "
            f"Skipping domain filter."
        )
        return rows

    return [
        r for r in rows
        if r.get("domain") in valid_domains
    ]

def project_fields(rows: list[dict], fields: list[str]) -> list[dict]:
    """Keep row_id + domain always (for traceability) plus only the requested fields."""
    if not fields:
        return rows
    return [
        {k: r[k] for k in ("row_id", "domain", *fields) if k in r}
        for r in rows
    ]


_COMPARATORS = {
    "less than": "<", "under": "<", "below": "<",
    "more than": ">", "greater than": ">", "above": ">", "over": ">",
    "equals": "==", "equal to": "==", "is": "==",
    "at least": ">=", "at most": "<=",
}

def apply_record_filter(rows: list[dict], filter_desc: str) -> list[dict]:
    """
    Lightweight mechanical parser for simple comparison filters like
    '<field> less than <value>' or '<field> equals <value>'.
    If the filter can't be confidently parsed, returns ALL rows unchanged
    (safe default -- never silently drops data based on a guess).
    """
    if not filter_desc or filter_desc.strip().lower() in ("all records", "none", "no filter"):
        return rows

    desc = filter_desc.lower()
    for phrase, op in _COMPARATORS.items():
        if phrase in desc:
            # crude field/value extraction: "<field> <phrase> <value>"
            parts = desc.split(phrase)
            if len(parts) != 2:
                continue
            field_part = parts[0].strip().replace(" ", "_")
            value_part = parts[1].strip().split()[0].strip(".,")

            # find the actual field name (allow partial match against row keys)
            sample = rows[0] if rows else {}
            matched_field = next(
                (k for k in sample.keys() if k.lower() in field_part or field_part in k.lower()),
                None,
            )
            if not matched_field:
                continue

            try:
                value = float(value_part)
            except ValueError:
                value = value_part.strip('"\'')

            def cmp(row_val, op=op, value=value):
                try:
                    rv = float(row_val)
                    if op == "<":  return rv < value
                    if op == ">":  return rv > value
                    if op == ">=": return rv >= value
                    if op == "<=": return rv <= value
                    if op == "==": return rv == value
                except (TypeError, ValueError):
                    if op == "==":
                        return str(row_val).lower() == str(value).lower()
                return False

            filtered = [r for r in rows if matched_field in r and cmp(r[matched_field])]
            return filtered

    # Could not parse -- safe fallback, include everything and let the
    # explainer reason over the unfiltered set rather than silently dropping rows.
    return rows


def compute_aggregate(rows: list[dict], computation: str, field: str) -> Optional[dict]:
    """Deterministic, code-level computation -- no LLM, no rounding surprises."""
    if not computation or computation == "none" or not field:
        return None

    values = []
    for r in rows:
        if field in r:
            try:
                values.append(float(r[field]))
            except (TypeError, ValueError):
                continue

    if not values:
        return {"computation": computation, "field": field, "result": None, "n": 0}

    if computation == "sum":      result = sum(values)
    elif computation == "average":result = sum(values) / len(values)
    elif computation == "count":  result = len(values)
    elif computation == "max":    result = max(values)
    elif computation == "min":    result = min(values)
    else:                         result = None   # "compare" handled by explainer over raw rows

    return {"computation": computation, "field": field, "result": result, "n": len(values)}

def run_funnel(rows: list[dict], plan: dict) -> dict:
    """
    Executes:
    domain -> record filter -> field projection -> aggregate
    """

    if not plan.get("is_specific"):
        return {
            "narrowed_rows": [],
            "aggregate": None,
            "skipped": True
        }

    domains = plan.get("domains") or []
    fields = plan.get("fields") or []
    record_filter = plan.get("record_filter") or "all records"
    computation = plan.get("computation") or "none"
    computation_field = plan.get("computation_field")

    print("\nDEBUG")
    print("Planner domains:", domains)
    print("Available domains:", {r["domain"] for r in rows})

    step1 = filter_by_domain(rows, domains)

    print("Rows after domain filter:", len(step1))

    step2 = apply_record_filter(step1, record_filter)
    step3 = project_fields(step2, fields)

    aggregate = compute_aggregate(
        step2,
        computation,
        computation_field
    )

    return {
        "narrowed_rows": step3,
        "aggregate": aggregate,
        "skipped": False,
        "funnel_trace": {
            "after_domain_filter": len(step1),
            "after_record_filter": len(step2),
            "after_field_projection": len(step3),
        },
    }

print("✅ Stage 2 functions defined: run_funnel() and helpers")


✅ Stage 2 functions defined: run_funnel() and helpers


In [20]:
EXPLAINER_SYSTEM_GENERAL = """You are a helpful financial assistant. Answer the user's
question directly using your general knowledge. This question does not require looking up
any specific dataset -- answer it the way you normally would."""

EXPLAINER_SYSTEM_SPECIFIC = """You are a data-grounded financial assistant. You have been
given a NARROWED SLICE of records (already filtered to what's relevant) and, if applicable,
a CODE-COMPUTED AGGREGATE (already calculated deterministically -- treat this number as
authoritative, do not recompute or second-guess it; just explain it).

NARROWED RECORDS:
{records}

COMPUTED AGGREGATE (if any):
{aggregate}

Answer the user's question using ONLY this data. Rules:
- If an aggregate is provided, state it clearly and explain what it means in context.
- Reference specific records (row_id or relevant field values) when useful for the answer.
- If the narrowed records don't contain enough information to fully answer, say so explicitly.
- Be concise and direct. Lead with the answer.
"""


## Stage 3 — Explainer (LLM call #2)
General question → answered with **no data attached** at all.
Specific question → answered using **only** the narrowed slice + any code-computed aggregate.

In [21]:
from groq import Groq
import json
import os

client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

def explain(
    question: str,
    plan: dict,
    funnel_result: dict,
    model: str = "llama-3.3-70b-versatile",
    max_tokens: int = 700
) -> str:

    if funnel_result["skipped"]:
        system_prompt = EXPLAINER_SYSTEM_GENERAL
    else:
        system_prompt = EXPLAINER_SYSTEM_SPECIFIC.format(
            records=json.dumps(
                funnel_result["narrowed_rows"],
                indent=2
            ),
            aggregate=(
                json.dumps(
                    funnel_result["aggregate"],
                    indent=2
                )
                if funnel_result["aggregate"]
                else "none"
            ),
        )

    try:

        completion = client.chat.completions.create(
            model=model,
            temperature=0,
            max_tokens=max_tokens,
            messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": question
                }
            ]
        )

        return completion.choices[0].message.content

    except Exception as e:

        return f"⚠️ Explainer API error: {e}"

## Orchestration — Run the Full Funnel

In [22]:
def ask(
    question: str,
    rows: list[dict] = None,
    schema: dict = None,
    verbose: bool = True
) -> dict:

    rows = rows if rows is not None else STRATEGIES
    schema = schema if schema is not None else SCHEMA

    # Convert strategies into funnel-compatible rows
    if rows and isinstance(rows[0], dict) and "domain" not in rows[0]:
        rows = [
            {
                "row_id": i,
                "domain": "strategy",
                **r
            }
            for i, r in enumerate(rows)
        ]

    if verbose:
        print(f"❓ Question: {question}\n")
        print("🧭 Stage 1 — Planning (question only, no data shown)...")

    plan = plan_query(question, schema)

    if verbose:
        print(f"   is_specific: {plan.get('is_specific')}")
        print(f"   reasoning:   {plan.get('reasoning', '')}")

        if plan.get("is_specific"):
            print(f"   domains:     {plan.get('domains')}")
            print(f"   fields:      {plan.get('fields')}")
            print(f"   filter:      {plan.get('record_filter')}")
            print(
                f"   computation: {plan.get('computation')} "
                f"on {plan.get('computation_field')}"
            )

    if plan.get("is_specific"):

        if verbose:
            print(
                "\n🔻 Stage 2 — Narrowing funnel "
                "(domain → fields → records → compute)..."
            )

        funnel_result = run_funnel(rows, plan)

        if verbose:
            t = funnel_result["funnel_trace"]

            print(
                f"   {len(rows)} rows → domain filter → "
                f"{t['after_domain_filter']} "
                f"→ record filter → {t['after_record_filter']} "
                f"→ field projection → "
                f"{t['after_field_projection']} rows"
            )

            if funnel_result["aggregate"]:
                print(f"   Computed: {funnel_result['aggregate']}")

    else:

        if verbose:
            print(
                "\n⏭️ Stage 2 — Skipped "
                "(general question, no data needed)"
            )

        funnel_result = {
            "narrowed_rows": [],
            "aggregate": None,
            "skipped": True
        }

    if verbose:
        print("\n💬 Stage 3 — Explaining...\n")

    answer = explain(question, plan, funnel_result)

    return {
        "question": question,
        "plan": plan,
        "funnel_result": funnel_result,
        "answer": answer
    }

## Demo 1 — Specific Question, With Computation
Planner should detect `is_specific=True`, `domain=credit_card`, a filter, and `computation=average` or `count`.

In [28]:
result1 = ask(
    "why should i go for strategy 1 and not 2 ",rows=STRATEGY_ROWS
)

print("\n" + "═"*65)
print("  FINAL ANSWER")
print("═"*65)

for line in textwrap.wrap(result1["answer"], 78):
    print(line)

❓ Question: why should i go for strategy 1 and not 2 

🧭 Stage 1 — Planning (question only, no data shown)...
   is_specific: True
   reasoning:   The question requires comparing two strategies, which involves analyzing dataset records.
   domains:     ['strategy']
   fields:      ['strategy_id', 'strategy_name', 'expected_return_pct', 'volatility_pct', 'max_drawdown_pct', 'retirement_success_probability', 'goal_success_probability']
   filter:      all records
   computation: compare on strategy_id

🔻 Stage 2 — Narrowing funnel (domain → fields → records → compute)...

DEBUG
Planner domains: ['strategy']
Available domains: {'strategy'}
Rows after domain filter: 5
   5 rows → domain filter → 5 → record filter → 5 → field projection → 5 rows
   Computed: {'computation': 'compare', 'field': 'strategy_id', 'result': None, 'n': 0}

💬 Stage 3 — Explaining...


═════════════════════════════════════════════════════════════════
  FINAL ANSWER
═══════════════════════════════════════════════════

## Demo 2 — Specific Question, No Aggregation (just filter/list)

In [ ]:

result2 = ask("Which mutual funds have a 1-year return above 20%?")

print("\n" + "═"*65)
print("  FINAL ANSWER")
print("═"*65)
for line in textwrap.wrap(result2["answer"], 78):
    print(line)


❓ Question: Which mutual funds have a 1-year return above 20%?

🧭 Stage 1 — Planning (question only, no data shown)...
   is_specific: True
   reasoning:   The question requires looking at actual data rows to find mutual funds with a specific return.
   domains:     ['investments']
   fields:      ['instrument', 'expected_return_pct']
   filter:      expected_return_pct greater than 20 and instrument equals Mutual Fund
   computation: none on None

🔻 Stage 2 — Narrowing funnel (domain → fields → records → compute)...
   5 rows → domain filter → 0 → record filter → 0 → field projection → 0 rows

💬 Stage 3 — Explaining...


═════════════════════════════════════════════════════════════════
  FINAL ANSWER
═════════════════════════════════════════════════════════════════
There is no data available to answer this question. The narrowed records are
empty, and there is no computed aggregate provided.


## Demo 3 — General Question (no data attached at all)
Planner should detect `is_specific=False` and Stage 2 should be skipped entirely —
the explainer answers with zero rows in context, like a normal LLM call.

In [ ]:

result3 = ask("What's generally considered a healthy debt-to-income ratio?")

print("\n" + "═"*65)
print("  FINAL ANSWER")
print("═"*65)
for line in textwrap.wrap(result3["answer"], 78):
    print(line)

print(f"\n  ℹ️  Rows attached to explainer: {len(result3['funnel_result']['narrowed_rows'])}  "
      f"(should be 0 — general question, no data needed)")


❓ Question: What's generally considered a healthy debt-to-income ratio?

🧭 Stage 1 — Planning (question only, no data shown)...
   is_specific: False
   reasoning:   The question is about general knowledge of debt-to-income ratio.

⏭️ Stage 2 — Skipped (general question, no data needed)

💬 Stage 3 — Explaining...


═════════════════════════════════════════════════════════════════
  FINAL ANSWER
═════════════════════════════════════════════════════════════════
A healthy debt-to-income (DTI) ratio is generally considered to be 36% or
less. This means that your total monthly debt payments, including credit
cards, loans, and mortgage or rent, should not exceed 36% of your gross
income.  To break it down further, here are some general guidelines:  * 36% or
less: Healthy DTI ratio, indicating that you have a good balance between debt
and income. * 37-42%: Manageable DTI ratio, but you may want to consider
reducing debt to free up more money for savings and other expenses. * 43-49%:
High DTI 

## Demo 4 — Cross-Domain Question
Planner should select multiple domains.

In [ ]:

result4 = ask("Compare interest rates on loans versus expense ratios on asset management funds.")

print("\n" + "═"*65)
print("  FINAL ANSWER")
print("═"*65)
for line in textwrap.wrap(result4["answer"], 78):
    print(line)

print(f"\n  ℹ️  Domains selected by planner: {result4['plan'].get('domains')}")


❓ Question: Compare interest rates on loans versus expense ratios on asset management funds.

🧭 Stage 1 — Planning (question only, no data shown)...
   is_specific: True
   reasoning:   The question requires comparing specific data from the debt_plan and investments domains.
   domains:     ['debt_plan', 'investments']
   fields:      ['interest_rate', 'instrument', 'allocation_pct']
   filter:      all records
   computation: compare on interest_rate and allocation_pct

🔻 Stage 2 — Narrowing funnel (domain → fields → records → compute)...
   5 rows → domain filter → 0 → record filter → 0 → field projection → 0 rows
   Computed: {'computation': 'compare', 'field': 'interest_rate and allocation_pct', 'result': None, 'n': 0}

💬 Stage 3 — Explaining...


═════════════════════════════════════════════════════════════════
  FINAL ANSWER
═════════════════════════════════════════════════════════════════
There is no data available to compare interest rates on loans versus expense
ratios on asse

## Funnel Trace Summary — How Much Was Narrowed, Per Question

In [ ]:

all_results = [result1, result2, result3, result4]

print(f"{'Question':<55}{'Specific':>9}{'Rows→Explainer':>16}")
print("─" * 80)
for r in all_results:
    q_short = (r["question"][:52] + "...") if len(r["question"]) > 52 else r["question"]
    n_rows = len(r["funnel_result"]["narrowed_rows"])
    is_spec = r["plan"]["is_specific"]
    print(f"{q_short:<55}{str(is_spec):>9}{n_rows:>16}")

print()
print(f"Total dataset size: {len(STRATEGIES)} rows")
print(f"General questions never touch the dataset at all — 0 rows, 0 narrowing cost.")
print(f"Specific questions only ever see the narrowed slice, never the full 150 rows.")


Question                                                Specific  Rows→Explainer
────────────────────────────────────────────────────────────────────────────────
Out of the strategies suppose i want to go aggressiv...     True               0
Which mutual funds have a 1-year return above 20%?          True               0
What's generally considered a healthy debt-to-income...    False               0
Compare interest rates on loans versus expense ratio...     True               0

Total dataset size: 5 rows
General questions never touch the dataset at all — 0 rows, 0 narrowing cost.
Specific questions only ever see the narrowed slice, never the full 150 rows.


---
## 📌 Production Notes

- **Two calls total, fixed cost regardless of question complexity** — this is the main efficiency win over a per-question bespoke pipeline. The planner is small (question only, ~500 output tokens). The explainer only ever sees the narrowed slice, never the full dataset.
- **`record_filter` parsing is intentionally simple** — it handles common comparison patterns (`less than`, `above`, `equals`, etc.) via lightweight string matching, not a full query language. If your real questions need richer filters (multi-condition, ranges, text search), either extend `apply_record_filter`'s phrase table, or have the planner emit a small structured filter object (`{{"field": "annual_fee", "op": "<", "value": 1000}}`) instead of plain language — more reliable to parse, costs nothing extra in the planning call.
- **Safe-fallback philosophy throughout**: if the planner call fails → defaults to `is_specific=True` over all domains (never silently answers from nothing when it should have looked at data). If `record_filter` can't be parsed → includes all rows rather than guessing and dropping data. If `computation_field` is missing → returns `aggregate=None`. Every failure mode degrades toward "show more, not less."
- **Schema discovery runs once per dataset load**, not per question — cheap, and it's what prevents the planner from inventing a domain or field name that doesn't actually exist in your data.
- **Cross-domain questions** (Demo 4) work because `domains` in the plan is a list — the funnel just unions matching rows across however many domains the planner names.
- **Connect your real data**: replace Cell 2 with your actual file load. The only requirement is that every row has a `domain` key — everything else (field names, value types) is discovered automatically by `discover_schema()`.
